# 05 - Educación (EPH)

Análisis educativo de la población a partir de la EPH (INDEC), usando los parquets
compilados por **00_preparacion_bases**.

**Contenido:**
1. Nivel educativo de la población adulta (`NIVEL_ED`), último trimestre.
2. Asistencia escolar por edad (`CH10`).
3. Alfabetismo (`CH09`) y establecimiento público vs. privado (`CH11`).
4. Evolución temporal: % con secundario completo o más, y tasa de analfabetismo.

**Variables (base Personas):** `NIVEL_ED` (nivel educativo), `CH06` (edad),
`CH09` (sabe leer/escribir), `CH10` (asiste), `CH11` (público/privado), `PONDERA`.
Ver `.claude/memoria_EPH.md`.

## 1. Setup (Colab)

In [ ]:
import sys, os, shutil, glob

REPO_URL = "https://github.com/santiagoriverti/analisis_EPH.git"
REPO_DIR = "/content/analisis_EPH"

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull -q
else:
    !git clone -q {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
sys.path.insert(0, REPO_DIR)

from google.colab import drive
drive.mount('/content/drive')

DRIVE_PROCESSED = "/content/drive/MyDrive/carga_EPH/processed"
PROCESSED_DIR = "/content/processed_local"
os.makedirs(PROCESSED_DIR, exist_ok=True)
for src in glob.glob(os.path.join(DRIVE_PROCESSED, "eph_T*.parquet")):
    dst = os.path.join(PROCESSED_DIR, os.path.basename(src))
    if not os.path.exists(dst):
        shutil.copy(src, dst)
n = len(glob.glob(os.path.join(PROCESSED_DIR, "eph_T*.parquet")))
print(f"Parquets locales listos en {PROCESSED_DIR}: {n} trimestres")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_panel, list_available_quarters, _period_tag

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (11, 6)

quarters = list_available_quarters()
ULTIMO = quarters[-1]
print("Último trimestre:", _period_tag(*ULTIMO))

def cargar(columnas, quarters=None):
    """Carga columnas de la base Personas, forzando a numérico las pedidas."""
    cols = list(dict.fromkeys(["PONDERA", *columnas]))
    df = load_panel(columns=cols, quarters=quarters, out_dir=PROCESSED_DIR)
    for c in cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

## 2. Nivel educativo de la población adulta (25+ años, último trimestre)

Se toma población de 25 años o más (edad en que la mayoría ya terminó su trayectoria
educativa formal). `NIVEL_ED` describe el máximo nivel alcanzado.

In [ ]:
NIVEL_ED = {1: "Primario incompleto", 2: "Primario completo", 3: "Secundario incompleto",
            4: "Secundario completo", 5: "Superior/univ. incompleto",
            6: "Superior/univ. completo", 7: "Sin instrucción", 9: "Ns/Nr"}
ORDEN = [7, 1, 2, 3, 4, 5, 6]

ed = cargar(["NIVEL_ED", "CH06"], quarters=[ULTIMO])
adultos = ed[ed["CH06"] >= 25]
dist = adultos.groupby("NIVEL_ED")["PONDERA"].sum()
dist = (dist / dist.sum() * 100)
dist = dist.reindex(ORDEN).dropna()
dist.index = dist.index.map(lambda c: NIVEL_ED.get(int(c), str(int(c))))

ax = dist.plot(kind="barh", color="#4C72B0")
ax.set_xlabel("% de la población 25+")
ax.set_title(f"Nivel educativo máximo (población 25+) - {_period_tag(*ULTIMO)}")
for i, v in enumerate(dist):
    ax.text(v + 0.3, i, f"{v:.1f}%", va="center")
plt.tight_layout()
plt.show()

## 3. Asistencia escolar por grupo de edad (último trimestre)

`CH10`: 1=asiste, 2=asistió pero no asiste, 3=nunca asistió. Se muestra el % que
**asiste actualmente** por grupo de edad.

In [ ]:
asi = cargar(["CH10", "CH06"], quarters=[ULTIMO])
asi = asi[asi["CH06"].between(3, 29)].copy()
bins = [3, 5, 12, 14, 17, 22, 29]
labels = ["3-4", "5-11", "12-13", "14-16", "17-21", "22-29"]
asi["ge"] = pd.cut(asi["CH06"], bins=bins, labels=labels, right=True, include_lowest=True)

def tasa_asiste(g):
    return g.loc[g["CH10"] == 1, "PONDERA"].sum() / g["PONDERA"].sum() * 100

tasa = asi.groupby("ge", observed=True).apply(tasa_asiste, include_groups=False)

ax = tasa.plot(kind="bar", color="#55A868")
ax.set_xlabel("Grupo de edad")
ax.set_ylabel("% que asiste")
ax.set_title(f"Tasa de asistencia escolar por edad - {_period_tag(*ULTIMO)}")
for i, v in enumerate(tasa):
    ax.text(i, v + 0.8, f"{v:.1f}%", ha="center")
plt.xticks(rotation=0)
plt.ylim(0, 105)
plt.tight_layout()
plt.show()

## 4. Establecimiento público vs. privado (entre quienes asisten)

`CH11`: 1=público, 2=privado (solo para quienes asisten, último trimestre).

In [ ]:
TIPO = {1: "Público", 2: "Privado", 9: "Ns/Nr"}
pp = cargar(["CH10", "CH11"], quarters=[ULTIMO])
pp = pp[pp["CH10"] == 1]  # asisten actualmente
dist = pp.dropna(subset=["CH11"]).groupby("CH11")["PONDERA"].sum()
dist = dist[dist.index.isin([1, 2])]
dist.index = dist.index.map(lambda c: TIPO[int(c)])
dist = (dist / dist.sum() * 100)

ax = dist.plot(kind="bar", color=["#4C72B0", "#C44E52"])
ax.set_ylabel("% de quienes asisten")
ax.set_title(f"Tipo de establecimiento - {_period_tag(*ULTIMO)}")
for i, v in enumerate(dist):
    ax.text(i, v + 0.8, f"{v:.1f}%", ha="center")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 5. Evolución temporal (serie 2017-2025)

Dos indicadores por trimestre, sobre población de 25+ años:
- **% con secundario completo o más** (`NIVEL_ED` ∈ {4, 5, 6}).
- **Tasa de analfabetismo** (`CH09` == 2, no sabe leer/escribir).

In [ ]:
todo = cargar(["NIVEL_ED", "CH09", "CH06"])
adu = todo[todo["CH06"] >= 25].copy()

def indicadores(g):
    w = g["PONDERA"]
    sec_mas = w[g["NIVEL_ED"].isin([4, 5, 6])].sum() / w.sum() * 100
    # Analfabetismo: sobre población con respuesta válida (CH09 en {1,2})
    val = g[g["CH09"].isin([1, 2])]
    analf = val.loc[val["CH09"] == 2, "PONDERA"].sum() / val["PONDERA"].sum() * 100
    return pd.Series({"secundario_completo_o_mas": sec_mas, "analfabetismo": analf})

evol = adu.groupby(["ANIO", "TRIMESTRE"]).apply(indicadores, include_groups=False)
evol.index = [f"{y}T{p}" for y, p in evol.index]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 9), sharex=True)
evol["secundario_completo_o_mas"].plot(ax=ax1, marker="o", ms=4, color="#4C72B0")
ax1.set_ylabel("%")
ax1.set_title("Población 25+ con secundario completo o más")

evol["analfabetismo"].plot(ax=ax2, marker="o", ms=4, color="#C44E52")
ax2.set_ylabel("%")
ax2.set_title("Tasa de analfabetismo (población 25+)")
ax2.set_xticks(range(len(evol.index)))
ax2.set_xticklabels(evol.index, rotation=90)
plt.tight_layout()
plt.show()
evol.round(2)

---
Notebook sobre los parquets de `00_preparacion_bases`. Variables de la base Personas,
ponderadas con `PONDERA`.